# Graphs & identification

Explicit `CausalDiagram`s enable identification (g-formula / back-door / front-door)
when possible, and otherwise route to partial-ID LPs. Always check `tight`.


In [ ]:
import numpy as np
from partialid import Assumptions, CausalDiagram, Query, Variables, bound

def softmax_P(V, seed=0):
    rng = np.random.default_rng(seed)
    keys = V.cells()
    logits = rng.normal(size=len(keys))
    w = np.exp(logits - logits.max())
    w = w / w.sum()
    return {k: float(v) for k, v in zip(keys, w)}


## Point identification via back-door

$Z$ confounds $X\to Y$ but is observed: $Z \to X$, $Z \to Y$, $X \to Y$.


In [ ]:
V = Variables.from_binary("Z", "X", "Y")
g = CausalDiagram.from_edges(
    ["Z", "X", "Y"],
    directed=[("Z", "X"), ("Z", "Y"), ("X", "Y")],
    bidirected=[],
)
P = softmax_P(V, seed=0)
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
r = bound(V, P, Q, Assumptions.from_diagram(g))
print(r.summary(Q))
assert r.width < 1e-9 and r.method.startswith("id/")


## Unidentified under maximal confounding (order-only)

Same variables, but only a causal order — Manski-style wide bounds.


In [ ]:
r_order = bound(V, P, Q, Assumptions.order("Z", "X", "Y"))
print(r_order.summary(Q))
print(f"order width {r_order.width:.4f} >> ID width {r.width:.4f}")


## Front-door

$X \to M \to Y$ with confounding $X \leftrightarrow Y$, no $X \to Y$ direct edge.


In [ ]:
V_fd = Variables.from_binary("X", "M", "Y")
g_fd = CausalDiagram.from_edges(
    ["X", "M", "Y"],
    directed=[("X", "M"), ("M", "Y")],
    bidirected=[("X", "Y")],
    maximal_confounding=False,
)
P_fd = softmax_P(V_fd, seed=1)
Q_fd = Query.interventional(outcome="Y", do={"X": 1}, value=1)
r_fd = bound(V_fd, P_fd, Q_fd, Assumptions.from_diagram(g_fd))
print(r_fd.summary(Q_fd))
print(f"method={r_fd.method}  tight={r_fd.tight}")


## When bounds may be outer (`tight=False`)

Graphs with multiple free c-components and no usable WL–WR cut can fall back to
a merged-latent **outer** LP. Always check `r.tight` in pipelines.


In [ ]:
import warnings

V_o = Variables.from_binary("X", "U", "V", "Y")
cells = V_o.cells(["X", "U", "V", "Y"])
rng = np.random.default_rng(1)
pvec = rng.dirichlet(np.ones(len(cells)))
P_o = {c: float(v) for c, v in zip(cells, pvec)}
Q_o = Query.interventional(outcome="Y", do={"X": 1}, value=1)
g_o = CausalDiagram.from_edges(
    ["X", "U", "V", "Y"],
    directed=[("X", "Y"), ("U", "Y"), ("V", "Y"), ("X", "U")],
    bidirected=[("X", "Y"), ("U", "V")],
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    r_o = bound(V_o, P_o, Q_o, Assumptions.from_diagram(g_o))
print(r_o.summary(Q_o))
print(f"tight={r_o.tight}  notes_contain_outer={'bounds_may_be_outer' in r_o.notes}")
print(f"warnings emitted: {len(caught)}")


## Exclusion atoms without a full diagram


In [ ]:
# Order + explicit missing edge (same spirit as a diagram exclusion)
a = Assumptions.order("Z", "X", "Y") & Assumptions.exclusion("Y", "Z")
r_ex = bound(V, P, Q, a)
print(r_ex.summary(Q))
